In [2]:
import pandas as pd
import numpy as np

products = pd.read_csv('products.csv')
aisles = pd.read_csv('aisles.csv')
departments = pd.read_csv('departments.csv')

temp = pd.merge(products, aisles, on='aisle_id', how='left')
temp.head()

,product_id,product_name,aisle_id,department_id,aisle
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes
1,2,All-Seasons Salt,104,13,spices seasonings
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation


In [3]:
df = pd.merge(temp, departments, on='department_id', how='left')
df.head()

,product_id,product_name,aisle_id,department_id,aisle,department
0,1,Chocolate Sandwich Cookies,61,19,cookies cakes,snacks
1,2,All-Seasons Salt,104,13,spices seasonings,pantry
2,3,Robust Golden Unsweetened Oolong Tea,94,7,tea,beverages
3,4,Smart Ones Classic Favorites Mini Rigatoni Wit...,38,1,frozen meals,frozen
4,5,Green Chile Anytime Sauce,5,13,marinades meat preparation,pantry


In [4]:
from sklearn.preprocessing import OneHotEncoder

encoder = OneHotEncoder(handle_unknown='ignore',sparse_output=False)

encoded_values = encoder.fit_transform( df[['aisle', 'department']])

new_cols = encoder.get_feature_names_out(['aisle', 'department'])

import pandas as pd

df_encoded = pd.DataFrame(
    encoded_values,
    columns=new_cols,
    index = df.index)

df_encoded = pd.concat([df['product_id'], df_encoded], axis=1)

df_encoded.tail()

,product_id,aisle_air fresheners candles,aisle_asian foods,aisle_baby accessories,aisle_baby bath body care,aisle_baby food formula,aisle_bakery desserts,aisle_baking ingredients,aisle_baking supplies decor,aisle_beauty,...,department_household,department_international,department_meat seafood,department_missing,department_other,department_pantry,department_personal care,department_pets,department_produce,department_snacks
49683,49684,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
49684,49685,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
49685,49686,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
49686,49687,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
49687,49688,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [5]:
# check if one hot encoded converted correctly or no 

df.loc[0]  

product_id                                1
product_name     Chocolate Sandwich Cookies
aisle_id                                 61
department_id                            19
aisle                         cookies cakes
department                           snacks
Name: 0, dtype: object

In [6]:
row = df_encoded[df_encoded['product_id'] == 1]
row

,product_id,aisle_air fresheners candles,aisle_asian foods,aisle_baby accessories,aisle_baby bath body care,aisle_baby food formula,aisle_bakery desserts,aisle_baking ingredients,aisle_baking supplies decor,aisle_beauty,...,department_household,department_international,department_meat seafood,department_missing,department_other,department_pantry,department_personal care,department_pets,department_produce,department_snacks
0,1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [7]:
active_cols = row.loc[:, row.iloc[0] == 1].columns
active_cols

#It selects only those columns where the value is 1 in that row and with .columns will return the column names
#can compare this w the og table

Index(['product_id', 'aisle_cookies cakes', 'department_snacks'], dtype='object')

In [8]:
order_prior = pd.read_csv('order_products__prior.csv')
order_prior.head()

,order_id,product_id,add_to_cart_order,reordered
0,2,33120,1,1
1,2,28985,2,1
2,2,9327,3,0
3,2,45918,4,1
4,2,30035,5,0


In [9]:
#this is reorder so itll tell us how popular is product

order_prior_reorder = order_prior.groupby(['product_id'])['reordered'].mean().reset_index()
order_prior_reorder.head()


#meaning: prod 1 is reordered 61% time so its a staple

,product_id,reordered
0,1,0.613391
1,2,0.133333
2,3,0.732852
3,4,0.446809
4,5,0.600000


In [10]:
#calc how many times a product occured so overall popularity of a product
#reorder will tell us a personal habit and what are chances itll be ordered again meanwhile freq tells us how popular it is overall

freq = order_prior.groupby(['product_id']).count().reset_index()
freq.drop(['add_to_cart_order','reordered'],axis=1,inplace=True)
freq.rename(columns={'order_id':'freq'},inplace=True)
freq.head()

#meaning: prod 1 appears in 1852 orders so v popular compared to prod 5

,product_id,freq
0,1,1852
1,2,90
2,3,277
3,4,329
4,5,15


In [11]:
order_prior_reorder_freq = pd.merge(order_prior_reorder,freq,on='product_id',how='left')
order_prior_reorder_freq.head()

,product_id,reordered,freq
0,1,0.613391,1852
1,2,0.133333,90
2,3,0.732852,277
3,4,0.446809,329
4,5,0.600000,15


In [12]:
#BELOW THIS IS FOR ADD TO CART
#thisll tell us that this product is important in the shopping flow so thus if theyre added in similar positions frequently theyre similar

avg_add_to_cart = order_prior.groupby(['product_id'])['add_to_cart_order'].mean().reset_index()
avg_add_to_cart

'''
meaning:
add_to_cart_order = avg position at which item was added to cart (1, 2, 3, ...) eg: prod 1 is usually added as 5.8th item in cart
so :
Lower number = added early (frequently purchased essentials)
Higher number = added late (impulse buys or uncommon items)
Products added at similar positions might be related!
'''

'\nmeaning:\nadd_to_cart_order = avg position at which item was added to cart (1, 2, 3, ...) eg: prod 1 is usually added as 5.8th item in cart\nso :\nLower number = added early (frequently purchased essentials)\nHigher number = added late (impulse buys or uncommon items)\nProducts added at similar positions might be related!\n'

In [13]:
order_prior_reorder_freq_atc = pd.merge(order_prior_reorder_freq,avg_add_to_cart,on='product_id',how='left')
order_prior_reorder_freq_atc.head()

,product_id,reordered,freq,add_to_cart_order
0,1,0.613391,1852,5.801836
1,2,0.133333,90,9.888889
2,3,0.732852,277,6.415162
3,4,0.446809,329,9.507599
4,5,0.600000,15,6.466667


In [14]:
df_encoded.columns

Index(['product_id', 'aisle_air fresheners candles', 'aisle_asian foods',
       'aisle_baby accessories', 'aisle_baby bath body care',
       'aisle_baby food formula', 'aisle_bakery desserts',
       'aisle_baking ingredients', 'aisle_baking supplies decor',
       'aisle_beauty',
       ...
       'department_household', 'department_international',
       'department_meat seafood', 'department_missing', 'department_other',
       'department_pantry', 'department_personal care', 'department_pets',
       'department_produce', 'department_snacks'],
      dtype='object', length=156)

In [15]:
#apply log otw freq is long tail so popular products will dominate, also if a>b before after log also log(a)>log(b) so relative popularity is kept. 

order_prior_reorder_freq_atc_log = order_prior_reorder_freq_atc.copy()

order_prior_reorder_freq_atc_log['freq'] = np.log1p(
    order_prior_reorder_freq_atc_log['freq']
)

order_prior_reorder_freq_atc_log.head()

,product_id,reordered,freq,add_to_cart_order
0,1,0.613391,7.524561,5.801836
1,2,0.133333,4.510860,9.888889
2,3,0.732852,5.627621,6.415162
3,4,0.446809,5.799093,9.507599
4,5,0.600000,2.772589,6.466667


In [16]:
#merge categorical (one hot encoded) and behavorial cols (reorder, add to cart etc etc)
df_final = pd.merge(
    order_prior_reorder_freq_atc_log,   
    df_encoded,
    on='product_id',
    how='left'
)

df_final.head()

,product_id,reordered,freq,add_to_cart_order,aisle_air fresheners candles,aisle_asian foods,aisle_baby accessories,aisle_baby bath body care,aisle_baby food formula,aisle_bakery desserts,...,department_household,department_international,department_meat seafood,department_missing,department_other,department_pantry,department_personal care,department_pets,department_produce,department_snacks
0,1,0.613391,7.524561,5.801836,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,0.133333,4.510860,9.888889,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,3,0.732852,5.627621,6.415162,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.446809,5.799093,9.507599,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,0.600000,2.772589,6.466667,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [17]:
#now we will scale numerical columns using standard scaler

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scale_cols = ['reordered','freq','add_to_cart_order']
df_final[scale_cols]=scaler.fit_transform(df_final[scale_cols])
df_final.head()

,product_id,reordered,freq,add_to_cart_order,aisle_air fresheners candles,aisle_asian foods,aisle_baby accessories,aisle_baby bath body care,aisle_baby food formula,aisle_bakery desserts,...,department_household,department_international,department_meat seafood,department_missing,department_other,department_pantry,department_personal care,department_pets,department_produce,department_snacks
0,1,1.186585,1.716408,-1.291815,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2,-1.120260,0.091535,0.310171,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0
2,3,1.760637,0.693650,-1.051412,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,4,0.386098,0.786101,0.160718,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5,1.122237,-0.845674,-1.031224,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0


In [18]:
df_final[scale_cols].describe()

,reordered,freq,add_to_cart_order
count,4.967700e+04,4.967700e+04,4.967700e+04
mean,-5.807121e-17,2.870663e-16,7.895396e-17
std,1.000010e+00,1.000010e+00,1.000010e+00
min,-1.760974e+00,-1.966829e+00,-3.173972e+00
25%,-7.611027e-01,-7.821696e-01,-5.768634e-01
50%,4.883371e-02,-1.241217e-01,-1.579598e-02
75%,7.825305e-01,6.596287e-01,4.934199e-01
max,2.761710e+00,4.704102e+00,1.720826e+01


In [19]:
#now well do truncated svd 
#SVD automatically learns hidden similarity patterns(latent patterns) from all columns and represents each product as a compact numerical vector.

#dont need this for svd its not a feature
X = df_final.drop(columns=['product_id'])

from sklearn.decomposition import TruncatedSVD
svd = TruncatedSVD(
    n_components=64,  #64 works for now can change later if results are weird
    random_state=42
)

X_emb = svd.fit_transform(X)
X_emb[0]

array([-2.34783015e+00, -4.85161843e-01, -4.28631569e-01,  1.00756445e+00,
       -4.14573743e-01, -1.18194536e-01,  2.31942944e-02,  2.45053119e-02,
       -5.17897169e-02, -1.93464833e-02,  1.94800403e-02,  4.82770763e-02,
        2.52757527e-02, -6.36969608e-02, -2.87548219e-02,  1.88997394e-02,
       -3.64683197e-03,  1.42332692e-02, -1.28541266e-02,  5.84305882e-03,
        1.28399167e-02, -2.48643819e-01,  4.73156394e-02, -4.13457976e-02,
       -8.66783938e-03, -9.10394109e-04, -6.06703606e-02, -5.06453001e-01,
       -5.71393325e-01,  3.68781764e-01, -2.45414917e-01, -3.14956371e-03,
        8.19932855e-03,  2.56748580e-02, -1.17888257e-02, -1.27355465e-01,
       -3.69412310e-02,  3.91911051e-03, -1.93143257e-02,  1.74892128e-05,
       -2.43075086e-02, -1.86134369e-03,  2.96802555e-03,  1.74604312e-03,
       -1.16879722e-02, -1.08875253e-02,  5.81876529e-03,  4.58136704e-03,
       -6.65808890e-03,  3.90307314e-03, -1.06140817e-02,  3.09492310e-03,
        1.73961461e-04, -

In [20]:
#then normalise it: Normalization means scaling a vector so that its length (magnitude) becomes 1, without changing its direction.
#Normalization ensures we only compare patterns, not magnitude.

from sklearn.preprocessing import normalize

X_emb = normalize(X_emb)

#check if normalization worked or no

import numpy as np

np.linalg.norm(X_emb[0])  #np.linalg.norm() → calculates the Euclidean length (magnitude) of the vector so if normalized itl be 1

1.0

In [21]:
#knn and cosine similarity 

from sklearn.neighbors import NearestNeighbors

# Initialize KNN with cosine similarity
knn = NearestNeighbors(n_neighbors=6, metric='cosine')  # 6 = 5 similar + self
knn.fit(X_emb)

# Find top similar products for product 1
distances, indices = knn.kneighbors([X_emb[1]])

print("Top similar product indices:", indices[0])
print("Distances:", distances[0])


Top similar product indices: [    1  1584 28376 25763 44710  6229]
Distances: [0.         0.0007945  0.00083905 0.00345346 0.00470361 0.00471718]


In [22]:
#check if correct output or no 

similar_indices = [28376 ,1584 ,43591 ,35225, 33232]

similar_product_ids = df_final.iloc[similar_indices]['product_id']
similar_product_ids


28376    28382
1584      1585
43591    43599
35225    35231
33232    33238
Name: product_id, dtype: int64

In [23]:
products.loc[
    products['product_id'].isin(similar_product_ids),
    ['product_id', 'product_name', 'aisle_id', 'department_id']
]

,product_id,product_name,aisle_id,department_id
1584,1585,Organic Poultry Seasoning,104,13
28381,28382,Organic Minced Garlic,104,13
33237,33238,Organic Arrowroot Powder,104,13
35230,35231,Organic Italian Seasoning,104,13
43598,43599,With Zest of Lemon Lemon Pepper,104,13


In [25]:
import joblib
import os

os.makedirs("models", exist_ok=True)

product_recommender_data = {
    "knn": knn,
    "X_emb": X_emb,
    "product_id_list": df_final['product_id'].tolist()
}

joblib.dump(product_recommender_data, "backend/models/product_recomm.joblib")

print("Saved successfully")

Saved successfully
